In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [145]:
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

PLOTS = "plots/"

In [118]:
df=pd.read_csv("joined_with_issues.csv", parse_dates=["Fecha"])
df=df.rename(columns={"Fecha": "Ticket Date"})

print("First five rows:")
df.head()

First five rows:


,ID Ticket,Ticket Date,Employee ID,Agent ID,Request Category,Issue Type,Severity,Priority,Resolution Time (Days),Satisfaction Rate,Full Name,Email,Year of Birth,Month of Birth,Day of Birth
0,GMLTER-8343486050,2019-01-21,1645,NaN,Software,IT Request,2 - Normal,2 - Mid,3,5.0,NaN,NaN,NaN,NaN,NaN
1,KDLEET-2843409265,2018-11-05,16,32.0,Hardware,IT Error,2 - Normal,0 - Unassiged,8,5.0,Silvia Morales,silvia.morales@fp20analytics.com,1980.0,6.0,16.0
2,KWRTSR-9742693502,2016-11-19,1570,15.0,System,IT Request,3 - Mayor,1 - Low,7,5.0,Galindo Guadalupe,guadalupe.galindo@fp20analytics.com,1995.0,6.0,16.0
3,KDLTSR-0542491979,2016-05-01,1847,39.0,System,IT Request,2 - Normal,0 - Unassiged,15,3.0,Jesus Contreras,jesus.contreras@fp20analytics.com,1983.0,10.0,20.0
4,SHRENR-0044004118,2020-06-22,1951,1.0,Login Access,IT Error,3 - Mayor,NaN,0,4.0,Mata Lucero,lucero.mata@fp20analytics.com,1989.0,4.0,28.0


In [119]:
print("Check dtypes:")
print(df.dtypes)

print("\nDataFrame shape:", df.shape)

Check dtypes:
ID Ticket                         object
Ticket Date               datetime64[ns]
Employee ID                        int64
Agent ID                         float64
Request Category                  object
Issue Type                        object
Severity                          object
Priority                          object
Resolution Time (Days)             int64
Satisfaction Rate                float64
Full Name                         object
Email                             object
Year of Birth                    float64
Month of Birth                   float64
Day of Birth                     float64
dtype: object

DataFrame shape: (98960, 15)


In [120]:
raw_snapshot = {
    "Resolution Time (Days)": {"mean": df["Resolution Time (Days)"].mean(),
                                "median": df["Resolution Time (Days)"].median()},
    "Satisfaction Rate": {"mean": df["Satisfaction Rate"].mean(),
                           "median": df["Satisfaction Rate"].median()},
}

In [121]:
print("\nSee the info:")
df.info()


See the info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98960 entries, 0 to 98959
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   ID Ticket               98960 non-null  object        
 1   Ticket Date             98960 non-null  datetime64[ns]
 2   Employee ID             98960 non-null  int64         
 3   Agent ID                74099 non-null  float64       
 4   Request Category        98960 non-null  object        
 5   Issue Type              98960 non-null  object        
 6   Severity                98960 non-null  object        
 7   Priority                92025 non-null  object        
 8   Resolution Time (Days)  98960 non-null  int64         
 9   Satisfaction Rate       86080 non-null  float64       
 10  Full Name               74099 non-null  object        
 11  Email                   74099 non-null  object        
 12  Year of Birth           74099 n

task 2

In [122]:
df_null=df.isnull().sum()
df_null_percent=(df.isnull().sum() / df.shape[0]) * 100

In [ ]:
#null percentage table for all columns
df_null_table=pd.DataFrame({"df_null_count":df_null,"df_null_percent":df_null_percent})
df_null_table=df_null_table.sort_values("df_null_percent",ascending=False)
print(df_null_table)

                        df_null_count  df_null_percent
Agent ID                        24861        25.122272
Full Name                       24861        25.122272
Email                           24861        25.122272
Year of Birth                   24861        25.122272
Month of Birth                  24861        25.122272
Day of Birth                    24861        25.122272
Satisfaction Rate               12880        13.015360
Priority                         6935         7.007882
ID Ticket                           0         0.000000
Ticket Date                         0         0.000000
Employee ID                         0         0.000000
Request Category                    0         0.000000
Issue Type                          0         0.000000
Severity                            0         0.000000
Resolution Time (Days)              0         0.000000


In [124]:
#columns exceed a 20% null rate
over_20p=df_null_table[df_null_table["df_null_percent"]>20]
print("\nColumns exceeding 20% null rate:")
print(over_20p)


Columns exceeding 20% null rate:
                df_null_count  df_null_percent
Agent ID                24861        25.122272
Full Name               24861        25.122272
Email                   24861        25.122272
Year of Birth           24861        25.122272
Month of Birth          24861        25.122272
Day of Birth            24861        25.122272


In [125]:
#columns below 20% nulls
below_20p=[ c for c in df.columns
           if pd.api.types.is_numeric_dtype(df[c]) and 0<df_null_table.loc[c,'df_null_percent']<20]
print("Numeric columns below 20%: ",below_20p)

Numeric columns below 20%:  ['Satisfaction Rate']


In [126]:
for col in df[below_20p]:
    df[col]=df[col].fillna(df[col].median())

df.isnull().sum()
    

ID Ticket                     0
Ticket Date                   0
Employee ID                   0
Agent ID                  24861
Request Category              0
Issue Type                    0
Severity                      0
Priority                   6935
Resolution Time (Days)        0
Satisfaction Rate             0
Full Name                 24861
Email                     24861
Year of Birth             24861
Month of Birth            24861
Day of Birth              24861
dtype: int64

task 3


In [127]:
duplicate_count=df.duplicated().sum()
print("\nDuplicate count: ",duplicate_count)

df=df.drop_duplicates()
df_null_percent_after_duplicate= (df.isnull().sum()/df.shape[0] * 100).round(3)

print(f"Rows removed: {duplicate_count}")

null_change=pd.DataFrame({"Before":df_null_percent, "After":df_null_percent_after_duplicate})
null_change["changed"] = null_change["Before"] != null_change["After"]
print("Change in column's null percentage:")
print(null_change)



Duplicate count:  1462
Rows removed: 1462
Change in column's null percentage:
                           Before   After  changed
ID Ticket                0.000000   0.000    False
Ticket Date              0.000000   0.000    False
Employee ID              0.000000   0.000    False
Agent ID                25.122272  23.999     True
Request Category         0.000000   0.000    False
Issue Type               0.000000   0.000    False
Severity                 0.000000   0.000    False
Priority                 7.007882   6.999     True
Resolution Time (Days)   0.000000   0.000    False
Satisfaction Rate       13.015360   0.000     True
Full Name               25.122272  23.999     True
Email                   25.122272  23.999     True
Year of Birth           25.122272  23.999     True
Month of Birth          25.122272  23.999     True
Day of Birth            25.122272  23.999     True


task 4


In [128]:
mem_before=df.memory_usage(deep=True).sum()
print(f"\nMemory usage BEFORE conversion: {mem_before / 1e6:.2f} MB")
df.head()


Memory usage BEFORE conversion: 48.15 MB


,ID Ticket,Ticket Date,Employee ID,Agent ID,Request Category,Issue Type,Severity,Priority,Resolution Time (Days),Satisfaction Rate,Full Name,Email,Year of Birth,Month of Birth,Day of Birth
0,GMLTER-8343486050,2019-01-21,1645,NaN,Software,IT Request,2 - Normal,2 - Mid,3,5.0,NaN,NaN,NaN,NaN,NaN
1,KDLEET-2843409265,2018-11-05,16,32.0,Hardware,IT Error,2 - Normal,0 - Unassiged,8,5.0,Silvia Morales,silvia.morales@fp20analytics.com,1980.0,6.0,16.0
2,KWRTSR-9742693502,2016-11-19,1570,15.0,System,IT Request,3 - Mayor,1 - Low,7,5.0,Galindo Guadalupe,guadalupe.galindo@fp20analytics.com,1995.0,6.0,16.0
3,KDLTSR-0542491979,2016-05-01,1847,39.0,System,IT Request,2 - Normal,0 - Unassiged,15,3.0,Jesus Contreras,jesus.contreras@fp20analytics.com,1983.0,10.0,20.0
4,SHRENR-0044004118,2020-06-22,1951,1.0,Login Access,IT Error,3 - Mayor,NaN,0,4.0,Mata Lucero,lucero.mata@fp20analytics.com,1989.0,4.0,28.0


In [129]:
df["Priority_level"]=df["Priority"].str.extract(r"^(\d+)")
df["Priority_level"]=pd.to_numeric(df["Priority_level"], errors="coerce")

df["Severity_level"]=df["Severity"].str.extract(r"^(\d+)")
df["Severity_level"]=pd.to_numeric(df["Severity_level"], errors="coerce")

print(df[
    ["Priority",
     "Priority_level",
     "Severity",
     "Severity_level"]
].drop_duplicates().head(8))

print(f"\npriority_level nulls: {df['Priority_level'].isnull().sum()} "
      f"({df['Priority_level'].isnull().mean()*100:.2f}%) -- inherited from 'Priority'")

         Priority  Priority_level    Severity  Severity_level
0         2 - Mid             2.0  2 - Normal               2
1   0 - Unassiged             0.0  2 - Normal               2
2         1 - Low             1.0   3 - Mayor               3
4             NaN             NaN   3 - Mayor               3
5         1 - Low             1.0  2 - Normal               2
6        3 - High             3.0  2 - Normal               2
8             NaN             NaN  2 - Normal               2
25        2 - Mid             2.0  4 - Urgent               4

priority_level nulls: 6824 (7.00%) -- inherited from 'Priority'


In [ ]:
df["Agent_age"]=df["Ticket Date"].dt.year - df["Year of Birth"]
print(f"agent_age nulls: {df['Agent_age'].isnull().sum()} "
      f"({df['Agent_age'].isnull().mean()*100:.2f}%) -- inherited from unassigned tickets")

#Converting  repetitive string columns to category dtype
for col in ["Request Category", "Issue Type", "Priority", "Severity"]:
    df[col]=df[col].astype("category")

mem_after=df.memory_usage(deep=True).sum()
print(f"\nMemory usage AFTER conversion: {mem_after / 1e6:.2f} MB")
print(f"Memory saved: {(mem_before - mem_after) / 1e6:.2f} MB "
      f"({(1 - mem_after/mem_before)*100:.1f}% reduction)")

agent_age nulls: 23399 (24.00%) -- inherited from unassigned tickets

Memory usage AFTER conversion: 28.34 MB
Memory saved: 19.81 MB (41.1% reduction)


In [135]:
df.info()
df.shape
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 97498 entries, 0 to 98958
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   ID Ticket               97498 non-null  object        
 1   Ticket Date             97498 non-null  datetime64[ns]
 2   Employee ID             97498 non-null  int64         
 3   Agent ID                74099 non-null  float64       
 4   Request Category        97498 non-null  category      
 5   Issue Type              97498 non-null  category      
 6   Severity                97498 non-null  category      
 7   Priority                90674 non-null  category      
 8   Resolution Time (Days)  97498 non-null  int64         
 9   Satisfaction Rate       97498 non-null  float64       
 10  Full Name               74099 non-null  object        
 11  Email                   74099 non-null  object        
 12  Year of Birth           74099 non-null  float64    

,ID Ticket,Ticket Date,Employee ID,Agent ID,Request Category,Issue Type,Severity,Priority,Resolution Time (Days),Satisfaction Rate,Full Name,Email,Year of Birth,Month of Birth,Day of Birth,Priority_level,Severity_level,Agent_age
0,GMLTER-8343486050,2019-01-21,1645,NaN,Software,IT Request,2 - Normal,2 - Mid,3,5.0,NaN,NaN,NaN,NaN,NaN,2.0,2,NaN
1,KDLEET-2843409265,2018-11-05,16,32.0,Hardware,IT Error,2 - Normal,0 - Unassiged,8,5.0,Silvia Morales,silvia.morales@fp20analytics.com,1980.0,6.0,16.0,0.0,2,38.0
2,KWRTSR-9742693502,2016-11-19,1570,15.0,System,IT Request,3 - Mayor,1 - Low,7,5.0,Galindo Guadalupe,guadalupe.galindo@fp20analytics.com,1995.0,6.0,16.0,1.0,3,21.0
3,KDLTSR-0542491979,2016-05-01,1847,39.0,System,IT Request,2 - Normal,0 - Unassiged,15,3.0,Jesus Contreras,jesus.contreras@fp20analytics.com,1983.0,10.0,20.0,0.0,2,33.0
4,SHRENR-0044004118,2020-06-22,1951,1.0,Login Access,IT Error,3 - Mayor,NaN,0,4.0,Mata Lucero,lucero.mata@fp20analytics.com,1989.0,4.0,28.0,NaN,3,31.0


task 5

In [ ]:
numeric_cols=["Resolution Time (Days)", "Satisfaction Rate", "Priority_level",
                 "Severity_level", "Agent_age"]
print("\ndf.describe() on numeric columns:")
print(df[numeric_cols].describe())

skew_vals=df[numeric_cols].skew().sort_values(key=abs, ascending=False)
print("\nSkewness per numeric column (sorted by |skew|):")
print(skew_vals)

most_skewed_col=skew_vals.index[0]
print(f"\nMost skewed column: '{most_skewed_col}' (skew = {skew_vals.iloc[0]:.3f})")


df.describe() on numeric columns:
       Resolution Time (Days)  Satisfaction Rate  Priority_level  \
count            97498.000000       97498.000000    90674.000000   
mean                 4.553150           4.215451        1.589541   
std                  4.365518           1.212688        1.254549   
min                  0.000000           1.000000        0.000000   
25%                  0.000000           4.000000        0.000000   
50%                  4.000000           5.000000        2.000000   
75%                  7.000000           5.000000        3.000000   
max                 21.000000           5.000000        3.000000   

       Severity_level     Agent_age  
count    97498.000000  74099.000000  
mean         2.047693     33.805409  
std          0.377096      7.534597  
min          0.000000     20.000000  
25%          2.000000     27.000000  
50%          2.000000     35.000000  
75%          2.000000     39.000000  
max          4.000000     49.000000  

Skewness 

task 6

In [136]:
for col in ["Resolution Time (Days)", "Satisfaction Rate"]:
    Q1=df[col].quantile(0.25)
    Q3=df[col].quantile(0.75)
    IQR=Q3 - Q1
    lower=Q1 - 1.5 * IQR
    upper=Q3 + 1.5 * IQR
    n_outliers=((df[col] < lower) | (df[col] > upper)).sum()
    print(f"\nColumn: {col}")
    print(f"  Q1={Q1}, Q3={Q3}, IQR={IQR}")
    print(f"  Lower bound={lower:.2f}, Upper bound={upper:.2f}")
    print(f"  Rows outside bounds: {n_outliers} ({n_outliers/df.shape[0]*100:.2f}%)")


Column: Resolution Time (Days)
  Q1=0.0, Q3=7.0, IQR=7.0
  Lower bound=-10.50, Upper bound=17.50
  Rows outside bounds: 258 (0.26%)

Column: Satisfaction Rate
  Q1=4.0, Q3=5.0, IQR=1.0
  Lower bound=2.50, Upper bound=6.50
  Rows outside bounds: 10379 (10.65%)


task 7

In [150]:
#Line plot - monthly ticket volume over time
monthly=df.set_index("Ticket Date").resample("ME").size()
plt.figure(figsize=(9,5))
plt.plot(monthly.index,monthly.values,color="#1f6f8b")
plt.title("Monthly IT Ticket Volume Over Time")
plt.xlabel("Month")
plt.ylabel("Ticket Count")
plt.tight_layout()
plt.savefig(PLOTS + "1_line_ticket_volume.png")
plt.close()
print("Saved: 1_line_ticket_volume.png")


#Bar chart - mean resolution time by request category
bar_data = df.groupby("Request Category", observed=True)["Resolution Time (Days)"].mean().sort_values(ascending=False)
plt.figure(figsize=(9, 5))
bar_data.plot.bar(color="#1f6f8b")
plt.title("Average Resolution Time by Request Category")
plt.xlabel("Request Category")
plt.ylabel("Average Resolution Time (Days)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOTS + "2_bar_resolution_by_category.png")
plt.close()
print("Saved: 2_bar_resolution_by_category.png")

#Histogram - most skewed column
plt.figure(figsize=(9, 5))
sns.histplot(df[most_skewed_col].dropna(), bins=20, color="#1f6f8b")
plt.title(f"Distribution of '{most_skewed_col}' (skew = {skew_vals.iloc[0]:.2f})")
plt.xlabel(most_skewed_col)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(PLOTS + "3_histogram_most_skewed.png")
plt.close()
print("Saved: 3_histogram_most_skewed.png")

#Scatter plot - priority_level vs resolution time (expected correlated)
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df.sample(5000, random_state=42), x="Priority_level",y="Resolution Time (Days)", alpha=0.25, color="#1f6f8b")
plt.title("Ticket Priority Level vs Resolution Time")
plt.xlabel("Priority Level (0=Unassigned ... 3=High)")
plt.ylabel("Resolution Time (Days)")
plt.tight_layout()
plt.savefig(PLOTS + "4_scatter_priority_resolution.png")
plt.close()
print("Saved: 4_scatter_priority_resolution.png")

#Box plot - resolution time split by request category
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Request Category", y="Resolution Time (Days)", color="#a8c9d6")
plt.title("Resolution Time Distribution by Request Category")
plt.xlabel("Request Category")
plt.ylabel("Resolution Time (Days)")
plt.tight_layout()
plt.savefig(PLOTS + "5_box_resolution_by_category.png")
plt.close()
print("Saved: 5_box_resolution_by_category.png")


Saved: 1_line_ticket_volume.png
Saved: 2_bar_resolution_by_category.png
Saved: 3_histogram_most_skewed.png
Saved: 4_scatter_priority_resolution.png
Saved: 5_box_resolution_by_category.png


task 8


In [ ]:
#Correlation heatmap
corr_matrix=df[numeric_cols].corr()
print("\nPearson correlation matrix:")
print(corr_matrix)

corr_unstack=corr_matrix.where(~np.eye(len(numeric_cols), dtype=bool)).unstack().dropna()
top_pair=corr_unstack.abs().idxmax()
top_val=corr_matrix.loc[top_pair]
print(f"\nHighest absolute correlation pair: {top_pair} = {top_val:.3f}")

plt.figure(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Correlation Heat Map (Pearson)")
plt.tight_layout()
plt.savefig(PLOTS + "6_correlation_heatmap.png")
plt.close()
print("Saved: 6_correlation_heatmap.png")


Pearson correlation matrix:
                        Resolution Time (Days)  Satisfaction Rate  \
Resolution Time (Days)                1.000000          -0.005908   
Satisfaction Rate                    -0.005908           1.000000   
Priority_level                       -0.199898          -0.000315   
Severity_level                       -0.040536           0.003458   
Agent_age                            -0.017134          -0.001121   

                        Priority_level  Severity_level  Agent_age  
Resolution Time (Days)       -0.199898       -0.040536  -0.017134  
Satisfaction Rate            -0.000315        0.003458  -0.001121  
Priority_level                1.000000        0.028122  -0.006043  
Severity_level                0.028122        1.000000  -0.002377  
Agent_age                    -0.006043       -0.002377   1.000000  

Highest absolute correlation pair: ('Resolution Time (Days)', 'Priority_level') = -0.200
Saved: 6_correlation_heatmap.png


task 9a

In [154]:
top2_skew_cols=skew_vals.index[:2].tolist()
print(f"\nTwo highest-|skew| numeric columns: {top2_skew_cols}")

for col in top2_skew_cols:
    if col in raw_snapshot:
        #This column already got median-filled in Task 2 (it had <20% nulls at load time)
        mean_val=raw_snapshot[col]["mean"]
        median_val=raw_snapshot[col]["median"]
        note=" (captured before Task 2's fillna -- this column was already imputed there)"
    else:
        mean_val=df[col].mean()
        median_val=df[col].median()
        note=""
    print(f"\nColumn: {col}{note}")
    print(f"  Mean   = {mean_val:.3f}")
    print(f"  Median = {median_val:.3f}")
    print(f"  Skew   = {skew_vals[col]:.3f}")
    print(f"  Nulls remaining right now: {df[col].isnull().sum()}")
    #applying median fill (no-op if Task 2 already handled it)
    df[col]=df[col].fillna(median_val)

print("\nNulls remaining in those columns after imputation:")
print(df[top2_skew_cols].isnull().sum())


Two highest-|skew| numeric columns: ['Severity_level', 'Satisfaction Rate']

Column: Severity_level
  Mean   = 2.048
  Median = 2.000
  Skew   = 1.697
  Nulls remaining right now: 0

Column: Satisfaction Rate (captured before Task 2's fillna -- this column was already imputed there)
  Mean   = 4.099
  Median = 5.000
  Skew   = -1.671
  Nulls remaining right now: 0

Nulls remaining in those columns after imputation:
Severity_level       0
Satisfaction Rate    0
dtype: int64


task 9b

In [155]:
spearman_matrix= df[numeric_cols].corr(method="spearman")
print("\nSpearman correlation matrix:")
print(spearman_matrix)

print("\nPearson correlation matrix (from Task 8):")
print(corr_matrix)

diff_matrix= (spearman_matrix - corr_matrix).abs()
seen,rows= set(), []
for a in numeric_cols:
    for b in numeric_cols:
        if a == b:
            continue
        key=frozenset([a, b])
        if key not in seen:
            seen.add(key)
            rows.append((a, b, diff_matrix.loc[a, b], spearman_matrix.loc[a, b], corr_matrix.loc[a, b]))

diff_table=pd.DataFrame(rows, columns=["col_a", "col_b", "abs_diff", "spearman", "pearson"])
diff_table=diff_table.sort_values("abs_diff", ascending=False)
print("\nTop 3 pairs by |Spearman - Pearson| difference:")
print(diff_table.head(3))


Spearman correlation matrix:
                        Resolution Time (Days)  Satisfaction Rate  \
Resolution Time (Days)                1.000000          -0.010601   
Satisfaction Rate                    -0.010601           1.000000   
Priority_level                       -0.142646           0.002461   
Severity_level                       -0.016968           0.013991   
Agent_age                            -0.015494           0.002198   

                        Priority_level  Severity_level  Agent_age  
Resolution Time (Days)       -0.142646       -0.016968  -0.015494  
Satisfaction Rate             0.002461        0.013991   0.002198  
Priority_level                1.000000        0.028658  -0.005951  
Severity_level                0.028658        1.000000  -0.000729  
Agent_age                    -0.005951       -0.000729   1.000000  

Pearson correlation matrix (from Task 8):
                        Resolution Time (Days)  Satisfaction Rate  \
Resolution Time (Days)             

task 9c

In [156]:
agg=df.groupby("Request Category", observed=True)["Resolution Time (Days)"].agg(["mean", "std", "count"])
agg=agg.sort_values("mean", ascending=False)
print("\nResolution Time (Days) aggregation by Request Category:")
print(agg)

highest_mean_cat=agg["mean"].idxmax()
highest_std_cat=agg["std"].idxmax()
mean_ratio=agg["mean"].max() / max(agg["mean"].min(), 1e-9)
print(f"\nHighest mean resolution time: {highest_mean_cat} ({agg['mean'].max():.2f} days)")
print(f"Highest std (variance): {highest_std_cat} (std={agg['std'].max():.2f} days)")
print(f"Ratio of highest group mean to lowest group mean: {mean_ratio:.2f}x")


Resolution Time (Days) aggregation by Request Category:
                      mean       std  count
Request Category                           
Hardware          7.625398  4.347524   9733
System            6.615609  4.018920  39002
Software          5.238733  3.487424  19570
Login Access      0.313808  0.706433  29193

Highest mean resolution time: Hardware (7.63 days)
Highest std (variance): Hardware (std=4.35 days)
Ratio of highest group mean to lowest group mean: 24.30x


task 10

In [157]:
df.to_csv("cleaned_data.csv", index=False)
print(f"\nSaved cleaned_data.csv with shape {df.shape}")
print("\nDONE.")


Saved cleaned_data.csv with shape (97498, 18)

DONE.
